In [1]:
import cx_Oracle
from tqdm import tqdm
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib

In [2]:
%matplotlib inline
sns.set_style('whitegrid')
from matplotlib import font_manager
# font_path='/usr/share/fonts/cjkuni-uming/uming.ttc'
font_path = '/usr/share/fonts/truetype/arphic/uming.ttc'
matplotlib.rcParams['font.family']=font_manager.FontProperties(fname=font_path).get_name()
matplotlib.rcParams['axes.unicode_minus']=False

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
%matplotlib inline

### 批量处理所有股票

In [9]:
# 获取现金流量表中的季报数据
connection = cx_Oracle.connect('wind', 'wind', '10.6.60.114:1521/wind')
cursor = connection.cursor()
sql = "select * from ASHARECASHFLOWHIS where  STATEMENT_TYPE in (408001000,408005000,408027000,408028000,408036000,408045000)  order by REPORT_PERIOD,ACTUAL_ANN_DT"
cursor.execute(sql)
columns = [col[0] for col in cursor.description]
results = cursor.fetchall()
df1 = pd.DataFrame(results, columns=columns)
cursor.close()
connection.close()

# 获取现金流量表中的包含更正和调整数据
connection = cx_Oracle.connect('wind', 'wind', '10.6.60.114:1521/wind')
cursor = connection.cursor()
sql = "select * from ASHARECASHFLOWHIS where STATEMENT_TYPE in (408001000,408004000,408050000,408029000,408031000,408037000,408046000)  order by REPORT_PERIOD,ACTUAL_ANN_DT"
cursor.execute(sql)
columns = [col[0] for col in cursor.description]
results = cursor.fetchall()
df3 = pd.DataFrame(results, columns=columns)
cursor.close()
connection.close()

In [10]:
start_date=df1['REPORT_PERIOD'].min()
end_date='20250626'
print(f"开始日期：{start_date},结束日期：{end_date}")

开始日期：20051231,结束日期：20250626


In [11]:
df1.shape

(252270, 122)

In [13]:
df3_deal=df3.drop(columns=['OBJECT_ID','S_INFO_COMPCODE','ANN_DT','CRNCY_CODE','STATEMENT_TYPE_WIND','OPDATE','OPMODE'])

In [14]:
df3_deal

,S_INFO_WINDCODE,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,NET_INCR_DEP_COB,NET_INCR_LOANS_CENTRAL_BANK,NET_INCR_FUND_BORR_OFI,CASH_RECP_PREM_ORIG_INCO,...,SPE_BAL_NETCASH_EQU_UNDIR,TOT_BAL_NETCASH_EQU_UNDIR,SPE_BAL_NETCASH_INC_UNDIR,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP
0,600000.SH,20020817,20010630,408004000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
1,600036.SH,20060809,20050630,408004000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
2,600000.SH,20060812,20050630,408004000,NaN,NaN,NaN,NaN,NaN,NaN,...,1.600604e+09,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
3,600015.SH,20060217,20051231,408001000,NaN,NaN,5.082396e+10,NaN,-2.196791e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
4,600016.SH,20060228,20051231,408001000,NaN,NaN,1.087043e+11,NaN,NaN,NaN,...,2.896320e+08,NaN,NaN,NaN,1.125393e+10,0.0,NaN,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
488450,301595.SZ,20250516,20250331,408001000,2.579398e+08,54817.30,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
488451,688300.SH,20250517,20250331,408001000,2.506897e+08,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
488452,603629.SH,20250614,20250331,408001000,1.177689e+09,4212193.89,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
488453,603356.SH,20250625,20250331,408001000,2.248219e+08,350291.41,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN


In [16]:
#获取所有股票代码
stock_ids=df3_deal['S_INFO_WINDCODE'].unique()
print(f"一共有{len(stock_ids)}个股票")

一共有5696个股票


In [17]:
df_test=df3_deal[df3_deal['S_INFO_WINDCODE']=='600000.SH']
df_test

,S_INFO_WINDCODE,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,NET_INCR_DEP_COB,NET_INCR_LOANS_CENTRAL_BANK,NET_INCR_FUND_BORR_OFI,CASH_RECP_PREM_ORIG_INCO,...,SPE_BAL_NETCASH_EQU_UNDIR,TOT_BAL_NETCASH_EQU_UNDIR,SPE_BAL_NETCASH_INC_UNDIR,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP
0,600000.SH,20020817,20010630,408004000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
2,600000.SH,20060812,20050630,408004000,NaN,NaN,NaN,NaN,NaN,NaN,...,1.600604e+09,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
5,600000.SH,20060302,20051231,408001000,NaN,NaN,NaN,NaN,NaN,NaN,...,2.688661e+09,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
12,600000.SH,20070324,20051231,408004000,NaN,NaN,NaN,NaN,NaN,NaN,...,2.688661e+09,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
1123,600000.SH,20060429,20060331,408001000,NaN,NaN,NaN,NaN,NaN,NaN,...,9.487801e+08,0.01,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
465670,600000.SH,20250430,20240331,408004000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
467350,600000.SH,20240820,20240630,408001000,NaN,NaN,NaN,NaN,NaN,NaN,...,-9.731000e+09,NaN,NaN,NaN,NaN,0.0,NaN,None,3.254600e+10,NaN
477417,600000.SH,20241031,20240930,408001000,NaN,NaN,1.379630e+11,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,None,NaN,NaN
478146,600000.SH,20250329,20241231,408001000,NaN,NaN,2.391400e+10,NaN,NaN,NaN,...,-2.448200e+10,NaN,NaN,NaN,NaN,0.0,NaN,None,6.943700e+10,NaN


In [33]:
def convert_windcode_to_symbol(windcode):
    """
    将wind代码转换为symbol格式
    例如：600000.sh -> sh600000
    """
    code,exchange=windcode.split('.')
    exchange_map={'SH':'sh','SZ':'sz'}
    
    return f"{exchange_map.get(exchange,exchange.lower())}{code}"

In [34]:
# 确定要处理的字段
fields = [col for col in df_test.columns 
          if col not in ['S_INFO_WINDCODE', 'ACTUAL_ANN_DT', 'REPORT_PERIOD', 'STATEMENT_TYPE']]
fields

['CASH_RECP_SG_AND_RS',
 'RECP_TAX_RENDS',
 'NET_INCR_DEP_COB',
 'NET_INCR_LOANS_CENTRAL_BANK',
 'NET_INCR_FUND_BORR_OFI',
 'CASH_RECP_PREM_ORIG_INCO',
 'NET_INCR_INSURED_DEP',
 'NET_CASH_RECEIVED_REINSU_BUS',
 'NET_INCR_DISP_TFA',
 'NET_INCR_INT_HANDLING_CHRG',
 'NET_INCR_DISP_FAAS',
 'NET_INCR_LOANS_OTHER_BANK',
 'NET_INCR_REPURCH_BUS_FUND',
 'OTHER_CASH_RECP_RAL_OPER_ACT',
 'STOT_CASH_INFLOWS_OPER_ACT',
 'CASH_PAY_GOODS_PURCH_SERV_REC',
 'CASH_PAY_BEH_EMPL',
 'PAY_ALL_TYP_TAX',
 'NET_INCR_CLIENTS_LOAN_ADV',
 'NET_INCR_DEP_CBOB',
 'CASH_PAY_CLAIMS_ORIG_INCO',
 'HANDLING_CHRG_PAID',
 'COMM_INSUR_PLCY_PAID',
 'OTHER_CASH_PAY_RAL_OPER_ACT',
 'STOT_CASH_OUTFLOWS_OPER_ACT',
 'NET_CASH_FLOWS_OPER_ACT',
 'CASH_RECP_DISP_WITHDRWL_INVEST',
 'CASH_RECP_RETURN_INVEST',
 'NET_CASH_RECP_DISP_FIOLTA',
 'NET_CASH_RECP_DISP_SOBU',
 'OTHER_CASH_RECP_RAL_INV_ACT',
 'STOT_CASH_INFLOWS_INV_ACT',
 'CASH_PAY_ACQ_CONST_FIOLTA',
 'CASH_PAID_INVEST',
 'NET_CASH_PAY_AQUIS_SOBU',
 'OTHER_CASH_PAY_RAL_INV_ACT',

In [98]:
result_data=[]

for _,row in df_test.iterrows():
    symbol=convert_windcode_to_symbol(row['S_INFO_WINDCODE'])
    date = row['ACTUAL_ANN_DT']
    period = row['REPORT_PERIOD']
    date = pd.to_datetime(date)
    period = pd.to_datetime(period)
    
    for field in fields:
        value = row[field]
        result_data.append({
            'date':date,
            'period':period,
            'value':value,
            'field':field,
            'symbol':symbol
        })

In [99]:
result_df=pd.DataFrame(result_data)


In [100]:
result_df.to_csv(f"qlib_data/pit/cashflow/{symbol}.csv",index=False)

In [101]:
result_df

,date,period,value,field,symbol
0,2002-08-17,2001-06-30,NaN,CASH_RECP_SG_AND_RS,sh600000
1,2002-08-17,2001-06-30,NaN,RECP_TAX_RENDS,sh600000
2,2002-08-17,2001-06-30,NaN,NET_INCR_DEP_COB,sh600000
3,2002-08-17,2001-06-30,NaN,NET_INCR_LOANS_CENTRAL_BANK,sh600000
4,2002-08-17,2001-06-30,NaN,NET_INCR_FUND_BORR_OFI,sh600000
...,...,...,...,...,...
17089,2025-04-30,2025-03-31,0.0,IS_CALCULATION,sh600000
17090,2025-04-30,2025-03-31,NaN,SECURITIE_NETCASH_RECEIVED,sh600000
17091,2025-04-30,2025-03-31,None,OTHER_IMPAIR_LOSS_ASSETS,sh600000
17092,2025-04-30,2025-03-31,NaN,CREDIT_IMPAIRMENT_LOSS,sh600000


In [102]:
class PitNormalize():

    def __init__(self, interval: str = "quarterly", *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.interval = interval

    def normalize(self, df: pd.DataFrame) -> pd.DataFrame:
        dt = df["period"].apply(
            lambda x: (
                pd.to_datetime(x) + pd.DateOffset(days=(45 if self.interval == 'quarterly' else 90))
            ).date()
        )
        df["date"] = df["date"].fillna(dt.astype(str))

        df["period"] = pd.to_datetime(df["period"])
        df["period"] = df["period"].apply(
            lambda x: x.year if self.interval == 'annual' else x.year * 100 + (x.month - 1) // 3 + 1
        )
        return df


In [103]:
#创建标准化器
normalizer=PitNormalize(interval='quarterly')

In [104]:
#标准化数据
normalized_df =normalizer.normalize(result_df)

In [105]:
normalized_df

,date,period,value,field,symbol
0,2002-08-17,200102,NaN,CASH_RECP_SG_AND_RS,sh600000
1,2002-08-17,200102,NaN,RECP_TAX_RENDS,sh600000
2,2002-08-17,200102,NaN,NET_INCR_DEP_COB,sh600000
3,2002-08-17,200102,NaN,NET_INCR_LOANS_CENTRAL_BANK,sh600000
4,2002-08-17,200102,NaN,NET_INCR_FUND_BORR_OFI,sh600000
...,...,...,...,...,...
17089,2025-04-30,202501,0.0,IS_CALCULATION,sh600000
17090,2025-04-30,202501,NaN,SECURITIE_NETCASH_RECEIVED,sh600000
17091,2025-04-30,202501,None,OTHER_IMPAIR_LOSS_ASSETS,sh600000
17092,2025-04-30,202501,NaN,CREDIT_IMPAIRMENT_LOSS,sh600000


In [ ]:
result_df.to_csv(f"qlib_data/pit_normalized/cashflow/{symbol}_normalized.csv",index=False)